# Lab 3: Modelo condicional para imagens

No último laboratório, vamos estudar a geração condicional de imagens, usando o dataset MNIST. Tal dataset contém, 32 x 32 = 1024 dimensões! Desse modo, tal conjunto de dados exigirá:

    1. O uso da classificação livre guiada (classifier-free - CFG, parte 2.1);
    
    2. Para a parametrização do vetor de aprendizagem, uma simples arquitetura MLP não será suficiente, exigindo assim, o uso de uma arquitetura consideravelmente mais robusto, que neste caso, será a U-Net (parte 2.2);

In [5]:
# Imports necessários
from abc import ABC, abstractmethod
from typing import Optional, List, Type, Tuple, Dict
import math

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.axes._axes import Axes
import torch
import torch.nn as nn
import torch.distributions as D
from tqdm import tqdm
import seaborn as sns
from sklearn.datasets import make_moons, make_circles
from torchvision import datasets, transforms
from torchvision.utils import make_grid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
# Reiclando classes dos laboratórios anteriores
class OldSampleAble(ABC):
    """
    Distribution which can be sampled from
    """
    @abstractmethod
    def sample(self, num_samples: int) -> torch.Tensor:
        """
        Args:
            - num_samples: the desired number of samples
        Returns:
            - samples: shape (batch_size, ...)
        """
        pass

O dataset MNIST possui ambos, imagens (números escritos) e as respectivas classes ('labels', de 0-9). Dessa forma, precisaremos adaptar a classe SampleAble, aqui representada acima como OldSampleAble, redefinindo-a para que retorne, não apenas as amostras, ``samples: torch.Tensor ``, mas também, as classes, ``labels: Optional[torch.Tensor]``. Assim sendo, estaremos formalizando, cada instância ``Sampleable``, como uma amostragem de uma distribuição conjunta ('joint distribution') sobre os dados e as classes. Implementamos esse novo  ``Sampleable`` abaixo

In [7]:
class Sampleable(ABC):
    """
    Distribution which can be sampled from
    """
    @abstractmethod
    def sample(self, num_samples: int) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        """
        Args:
            - num_samples: the desired number of samples
        Returns:
            - samples: shape (batch_size, ...)
            - labels: shape(batch_size, label_dim)
        """
        pass

Para certas distribuições, como a Gaussiana, não faz muito sentido pensar em termos de classes. Por essa razão, fizemos com que o paramêtro labels retornasse o valor ``Optional``: uma gaussiana que retorna ``None``.  Abaixo, a classe ``IsotropicGaussian`` é implementada

In [9]:
class IsotropicGaussian(nn.Module, Sampleable):
    """
    Sampleable wrapper around torch.randn
    """
    def __init__(self, shape: List[int], std: float = 1.0):
        """
        shape: shape of sampled data
        """
        super().__init__()
        self.shape = shape
        self.std = std
        self.dummy = nn.Buffer(torch.zeros(1)) # Will automatically be moved when self.to(...) is called.

    def sample(self, num_samples: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.std * torch.randn(num_samples, *self.shape).to(self.dummy.device), None